# Time Travel в DuckLake

DuckLake хранит историю всех транзакций в PostgreSQL-каталоге.
Это позволяет:
- Читать данные на любой момент времени (`AT TIMESTAMP`)
- Читать данные по номеру снэпшота (`AT VERSION`)
- Сравнивать состояние данных между двумя точками
- Восстанавливать случайно удалённые данные

In [ ]:
import os, sys
sys.path.insert(0, '..')

os.environ.setdefault('DUCKLAKE_PG_HOST', 'localhost')
os.environ.setdefault('DUCKLAKE_PG_DB', 'ducklake_catalog')
os.environ.setdefault('DUCKLAKE_PG_USER', 'ducklake')
os.environ.setdefault('DUCKLAKE_PG_PASSWORD', 'ducklake_secret_change_me')
os.environ.setdefault('RUSTFS_ACCESS_KEY', 'rustfsadmin')
os.environ.setdefault('RUSTFS_SECRET_KEY', 'rustfsadmin123')
os.environ.setdefault('RUSTFS_ENDPOINT', 'http://localhost:9000')
os.environ.setdefault('RUSTFS_BUCKET', 'ducklake-flights')

from src.generators.connection import get_ducklake_connection
conn = get_ducklake_connection()
print('Connected!')

## Просмотр всех снэпшотов

In [ ]:
import pandas as pd

snapshots = conn.execute("""
    SELECT snapshot_id, snapshot_time, schema_version
    FROM ducklake_snapshots('flights')
    ORDER BY snapshot_id
""").df()

print(f'Всего снэпшотов: {len(snapshots)}')
print(f'Первый: {snapshots.snapshot_time.min()}')
print(f'Последний: {snapshots.snapshot_time.max()}')
snapshots.tail(10)

## Чтение данных на конкретный момент времени

In [ ]:
# Берём два снэпшота для сравнения
if len(snapshots) >= 2:
    snap_first = int(snapshots.iloc[0]['snapshot_id'])
    snap_last = int(snapshots.iloc[-1]['snapshot_id'])

    count_first = conn.execute(f"""
        SELECT COUNT(*) FROM flights.bookings AT (VERSION => {snap_first})
    """).fetchone()[0]

    count_last = conn.execute(f"""
        SELECT COUNT(*) FROM flights.bookings AT (VERSION => {snap_last})
    """).fetchone()[0]

    print(f'Снэпшот #{snap_first}: {count_first:,} бронирований')
    print(f'Снэпшот #{snap_last}: {count_last:,} бронирований')
    print(f'Прирост: +{count_last - count_first:,}')
else:
    print('Нужно минимум 2 снэпшота. Запустите генератор.')

## Сравнение выручки между снэпшотами

In [ ]:
if len(snapshots) >= 2:
    # Топ маршрутов в первом снэпшоте
    df_first = conn.execute(f"""
        SELECT
            src_airport_iata || '-' || dst_airport_iata AS route,
            COUNT(*) AS bookings,
            ROUND(SUM(price_rub), 0) AS revenue
        FROM flights.bookings AT (VERSION => {snap_first})
        JOIN flights.flights USING (flight_id)
        GROUP BY 1
        ORDER BY revenue DESC
        LIMIT 5
    """).df()

    # Топ маршрутов в последнем снэпшоте
    df_last = conn.execute(f"""
        SELECT
            src_airport_iata || '-' || dst_airport_iata AS route,
            COUNT(*) AS bookings,
            ROUND(SUM(price_rub), 0) AS revenue
        FROM flights.bookings AT (VERSION => {snap_last})
        JOIN flights.flights USING (flight_id)
        GROUP BY 1
        ORDER BY revenue DESC
        LIMIT 5
    """).df()

    print(f'Топ-5 маршрутов в снэпшоте #{snap_first}:')
    display(df_first)
    print(f'Топ-5 маршрутов в снэпшоте #{snap_last}:')
    display(df_last)

## Time travel по метке времени

In [ ]:
from datetime import datetime, timedelta, timezone

# Данные час назад
one_hour_ago = (datetime.now(timezone.utc) - timedelta(hours=1)).isoformat()

try:
    count = conn.execute(f"""
        SELECT COUNT(*)
        FROM flights.bookings AT (TIMESTAMP => TIMESTAMPTZ '{one_hour_ago}')
    """).fetchone()[0]
    print(f'Бронирований час назад: {count:,}')
except Exception as e:
    print(f'Ошибка time travel по timestamp: {e}')

# Текущее количество
count_now = conn.execute("SELECT COUNT(*) FROM flights.bookings").fetchone()[0]
print(f'Бронирований сейчас:    {count_now:,}')

In [ ]:
conn.close()
print('Done.')